# Dashboard para Monitoramento de Jogos Digitais

### Objetivo Geral

1. Oferecer uma visão centralizada e intuitiva dos principais indicadores de desempenho dos jogos (como pico de jogadores simultâneos, tempo de jogo, preço e avaliações).
2. Avaliar tanto o engajamento (playtime, playtime mediano) quanto a satisfação dos usuários (`pct_pos_total`, `recommendations`) em tempo real.
3. Mensurar o impacto de estratégias comerciais (preço e descontos) sobre o comportamento do jogador e receita estimada.
4. Identificar oportunidades de loteamento e crescimento por plataforma, idioma, DLCs e temas mais populares (tags).
5. Apoiar decisões táticas e estratégicas das equipes de produto, marketing e infraestrutura através de comparações, benchmarks e insights acionáveis.

### Público‑alvo

O dashboard é voltado para:

* **Gerentes de Produto e Marketing**: para entender o que impacta o engajamento e monetização, identificando títulos com potencial para promoção ou investimento adicional;
* **Analistas de Dados e Business Intelligence**: para encontrar padrões, outliers e tendências na utilização, reviews e vendas;
* **Equipes de Infraestrutura e Operações**: para monitorar pico de uso (`peak_ccu`), planejar capacidade de servidores e evitar falhas nos lançamentos;
* **Executivos e Stakeholders**: que precisam de uma visão consolidada e visual do desempenho do portfólio, para embasar decisões estratégicas com dados confiáveis.

Com base nas ideias acima, as colunas que serão priorizadas serão:


1. **appid** (int64):
   ID único atribuído pelo Steam a cada jogo. Útil para cruzamentos com APIs ou bases externas.

2. **name** (object):
   Nome do jogo/título da aplicação.

3. **release\_date** (datetime64\[ns]):
   Data oficial de lançamento do jogo na plataforma Steam.

4. **required\_age** (int64):
   Idade mínima recomendada/necessária para jogar, conforme classificação etária da Steam.

5. **price** (float64):
   Preço atual do jogo, geralmente em dólares. Valores típicos variam de 0 (grátis) até cerca de 59,99.

6. **dlc\_count** (int64):
   Quantidade de DLCs (conteúdo adicional pago) associados ao jogo.

7. **header\_image** (object):
   URL da imagem de cabeçalho/banner do jogo, usada na interface do Steam.

8. **windows** (bool):
   Indica se o jogo tem suporte ao sistema operacional Windows (True/False).

9. **mac** (bool):
   Indica se o jogo tem suporte ao macOS.

10. **linux** (bool):
    Indica se o jogo tem suporte ao Linux.

11. **metacritic\_score** (int64):
    Nota agregada do Metacritic (0–100), refletindo avaliações da mídia especializada.

12. **recommendations** (int64):
    Total de recomendações positivas de usuários no Steam — geralmente representando quantos "curtiram" o jogo.

13. **supported\_languages** (object):
    Lista de idiomas suportados na interface do jogo (legendas e UI).

14. **full\_audio\_languages** (object):
    Idiomas disponíveis com áudio completo no jogo (voltações).

15. **publishers** (object):
    Empresa(s) publicadoras responsáveis pela distribuição do jogo.

16. **positive** (int64):
    Número bruto de análises positivas (reviews) dos usuários.

17. **negative** (int64):
    Número bruto de análises negativas dos usuários.

18. **average\_playtime\_forever** (int64):
    Tempo médio de jogo (em minutos) por todos os usuários que jogaram.

19. **median\_playtime\_forever** (int64):
    Tempo mediano de jogo (minutos) por todos os usuários — menos sensível a valores extremos.

20. **discount** (int64):
    Percentual de desconto atual aplicado (0 se não estiver em promoção, por exemplo, 50 para metade do preço).

21. **peak\_ccu** (int64):
    *Pico de usuários simultâneos* — maior número de jogadores online ao mesmo tempo durante a existência do jogo.

22. **tags** (object):
    Conjunto de etiquetas/temas atribuídos pelos usuários (ex.: `"Action;Multiplayer;Open World"`).

23. **pct\_pos\_total** (int64):
    Percentual de avaliações positivas em relação ao total de análises:

    ```
    pct_pos_total = (positive / (positive + negative)) * 100  
    ```

    Representa a proporção de reviews positivas — um indicador direto da satisfação geral dos jogadores.

24. **num\_reviews\_total** (int64):
    Soma total de avaliações: `positive + negative`.


### Resumo dos dados

| Coluna                                                          | Utilidade                                                            |
| --------------------------------------------------------------- | -------------------------------------------------------------------- |
| **price**, **discount**                                         | Avaliar impacto de promoções sobre vendas                            |
| **dlc\_count**, **tags**                                        | Analisar diversidade de conteúdo e segmentos de interesse            |
| **peak\_ccu**                                                   | Medir popularidade em tempo real & dimensionamento de infraestrutura |
| **pct\_pos\_total**, **recommendations**, **metacritic\_score** | Avaliar qualidade percebida e reputação geral                        |
| **average/median playtime**                                     | Entender engajamento e retenção do jogador

## Bibliotecas

In [60]:
!pip install rapidfuzz

                                              0.0/1.7 MB ? eta -:--:--
                                              0.0/1.7 MB ? eta -:--:--
                                              0.0/1.7 MB ? eta -:--:--
                                              0.0/1.7 MB 145.2 kB/s eta 0:00:12
                                              0.0/1.7 MB 163.4 kB/s eta 0:00:10
     -                                        0.1/1.7 MB 233.8 kB/s eta 0:00:07
     --                                       0.1/1.7 MB 364.4 kB/s eta 0:00:05
     ---                                      0.2/1.7 MB 437.1 kB/s eta 0:00:04
     ----                                     0.2/1.7 MB 491.5 kB/s eta 0:00:03
     ------                                   0.3/1.7 MB 561.1 kB/s eta 0:00:03
     ------                                   0.3/1.7 MB 589.5 kB/s eta 0:00:03
     --------                                 0.4/1.7 MB 674.7 kB/s eta 0:00:02
     ----------                               0.4/1.7 MB 763.9 kB/s

In [61]:
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz

## Leitura dos Dados

In [4]:
dados = pd.read_csv("../data/raw/games_march2025_cleaned.csv")

In [5]:
dados.release_date = pd.to_datetime(dados.release_date)

In [6]:
dados_acima_2024 = dados.query("release_date.dt.year >= 2024").copy()

In [7]:
dados_acima_2024.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 21855 entries, 10 to 89616
Data columns (total 47 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   appid                     21855 non-null  int64         
 1   name                      21855 non-null  object        
 2   release_date              21855 non-null  datetime64[ns]
 3   required_age              21855 non-null  int64         
 4   price                     21855 non-null  float64       
 5   dlc_count                 21855 non-null  int64         
 6   detailed_description      21791 non-null  object        
 7   about_the_game            21781 non-null  object        
 8   short_description         21812 non-null  object        
 9   reviews                   1157 non-null   object        
 10  header_image              21855 non-null  object        
 11  website                   6974 non-null   object        
 12  support_url      

In [8]:
dados_acima_2024.head()

,appid,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent
10,2358720,Black Myth: Wukong,2024-08-19,13,59.99,2,Digital Deluxe Edition The Black Myth: Wukong ...,Black Myth: Wukong is an action RPG rooted in ...,Black Myth: Wukong is an action RPG rooted in ...,NaN,...,0,0,0,0,35990,"{'Mythology': 9421, 'Action RPG': 7720, 'Actio...",96,825621,94,6139
15,553850,HELLDIVERS™ 2,2024-02-08,17,39.99,1,Digital Deluxe Edition Edition includes: ‘DP-5...,The Galaxy’s Last Line of Offence. Enlist in t...,The Galaxy’s Last Line of Offence. Enlist in t...,NaN,...,0,0,0,0,48290,"{'Online Co-Op': 939, 'PvE': 827, 'Third-Perso...",76,716489,90,8172
41,239140,Dying Light,2025-01-27,17,19.99,41,DYING LIGHT: THE BEAST Kyle Crane returns! The...,Dying Light Standard 10th Anniversary Edition ...,First-person action survival game set in a pos...,“Dying Light gradually and gratifyingly evolve...,...,0,0,0,0,6198,"{'Zombies': 6010, 'Survival Horror': 5732, 'Ho...",95,332069,94,2495
42,1623730,Palworld,2024-01-18,0,22.49,1,"Q. What kind of game is this? A. In this game,...","Q. What kind of game is this? A. In this game,...","Fight, farm, build and work alongside mysterio...",NaN,...,0,0,0,25,33835,"{'Open World': 1370, 'Survival': 1264, 'Creatu...",94,314689,92,3602
59,251570,7 Days to Die,2024-07-25,0,29.24,1,HOW LONG WILL YOU SURVIVE? With over 18 millio...,HOW LONG WILL YOU SURVIVE? With over 18 millio...,7 Days to Die is an open-world game that is a ...,“This game is the best Zombie FPS that I have ...,...,1358,1633,861,35,25426,"{'Survival': 6898, 'Zombies': 5233, 'Multiplay...",88,249049,86,1825


## Seleção de colunas

In [9]:
limite = dados_acima_2024.shape[0]/2
for coluna in dados_acima_2024.columns:
    if dados_acima_2024[coluna].isnull().sum() >= limite:
        dados_acima_2024.drop([coluna], axis=1, inplace=True)

In [10]:
dados_acima_2024.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 21855 entries, 10 to 89616
Data columns (total 41 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   appid                     21855 non-null  int64         
 1   name                      21855 non-null  object        
 2   release_date              21855 non-null  datetime64[ns]
 3   required_age              21855 non-null  int64         
 4   price                     21855 non-null  float64       
 5   dlc_count                 21855 non-null  int64         
 6   detailed_description      21791 non-null  object        
 7   about_the_game            21781 non-null  object        
 8   short_description         21812 non-null  object        
 9   header_image              21855 non-null  object        
 10  support_email             19451 non-null  object        
 11  windows                   21855 non-null  bool          
 12  mac              

In [11]:
dados_acima_2024.packages

10       [{'title': 'Buy Black Myth: Wukong', 'descript...
15       [{'title': 'Buy HELLDIVERS™ 2', 'description':...
41       [{'title': 'Buy Dying Light', 'description': '...
42       [{'title': 'Buy Palworld', 'description': '', ...
59       [{'title': 'Buy 7 Days to Die', 'description':...
                               ...                        
89600    [{'title': 'Buy Haunted Property', 'descriptio...
89602    [{'title': 'Buy Contraptions Collection', 'des...
89608    [{'title': 'Buy Caribbean Pirates', 'descripti...
89613    [{'title': 'Buy Outrun Them', 'description': '...
89616    [{'title': 'Buy DragonRoad', 'description': ''...
Name: packages, Length: 21855, dtype: object

In [12]:
dados_acima_2024.developers

10                 ['Game Science']
15       ['Arrowhead Game Studios']
41                     ['Techland']
42                   ['Pocketpair']
59                ['The Fun Pimps']
                    ...            
89600              ['Markus Korda']
89602         ['Moragami Co., Ltd']
89608                   ['Seaward']
89613                   ['TheBean']
89616                     ['XuJie']
Name: developers, Length: 21855, dtype: object

In [13]:
dados_acima_2024.publishers

10                            ['Game Science']
15              ['PlayStation Publishing LLC']
41                                ['Techland']
42                              ['Pocketpair']
59         ['The Fun Pimps Entertainment LLC']
                         ...                  
89600                         ['Markus Korda']
89602    ['Funbox Media Ltd', 'GS2 Games Inc']
89608                     ['BlackMark Studio']
89613                              ['TheBean']
89616                                ['XuJie']
Name: publishers, Length: 21855, dtype: object

In [14]:
dados_acima_2024.supported_languages       

10       ['English', 'French', 'German', 'Spanish - Spa...
15       ['English', 'French', 'Italian', 'German', 'Sp...
41       ['English', 'French', 'Italian', 'German', 'Sp...
42       ['English', 'Simplified Chinese', 'Traditional...
59       ['English', 'French', 'German', 'Spanish - Spa...
                               ...                        
89600                                ['English', 'German']
89602    ['English', 'French', 'Italian', 'German', 'Sp...
89608    ['English', 'Spanish - Spain', 'Russian', 'Tur...
89613                                          ['English']
89616    ['English', 'Simplified Chinese', 'French', 'G...
Name: supported_languages, Length: 21855, dtype: object

In [15]:
dados_acima_2024.recommendations           

10       825294
15       716326
41       336797
42       314601
59       248708
          ...  
89600         0
89602         0
89608         0
89613         0
89616         0
Name: recommendations, Length: 21855, dtype: int64

In [16]:
dados_acima_2024.user_score.value_counts()

0    21855
Name: user_score, dtype: int64

In [17]:
dados_acima_2024.positive.value_counts()

0       8526
1       1665
2       1181
3        928
4        703
        ... 
1681       1
1706       1
1231       1
1684       1
1122       1
Name: positive, Length: 1247, dtype: int64

In [18]:
dados_acima_2024.estimated_owners.value_counts()

0 - 20000               15855
0 - 0                    4207
20000 - 50000             908
50000 - 100000            367
100000 - 200000           246
200000 - 500000           161
500000 - 1000000           59
2000000 - 5000000          22
1000000 - 2000000          18
5000000 - 10000000          5
50000000 - 100000000        3
10000000 - 20000000         3
20000000 - 50000000         1
Name: estimated_owners, dtype: int64

In [19]:
colunas_descricao = ["detailed_description", "about_the_game", "short_description"]
colunas_links_suporte = ["support_email"]
colunas_acompanhamento = ["average_playtime_2weeks", "median_playtime_2weeks", "pct_pos_recent", "num_reviews_recent"] 
colunas_infos = ['achievements', 'packages', 'developers', 'categories', 'genres', 'screenshots', 'movies', 'user_score', 'estimated_owners']

In [20]:
colunas_dropar = [colunas_descricao, colunas_links_suporte, colunas_acompanhamento, colunas_infos]
for conj in colunas_dropar:
    dados_acima_2024.drop(conj, axis=1, inplace=True)

In [21]:
dados_acima_2024.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 21855 entries, 10 to 89616
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   appid                     21855 non-null  int64         
 1   name                      21855 non-null  object        
 2   release_date              21855 non-null  datetime64[ns]
 3   required_age              21855 non-null  int64         
 4   price                     21855 non-null  float64       
 5   dlc_count                 21855 non-null  int64         
 6   header_image              21855 non-null  object        
 7   windows                   21855 non-null  bool          
 8   mac                       21855 non-null  bool          
 9   linux                     21855 non-null  bool          
 10  metacritic_score          21855 non-null  int64         
 11  recommendations           21855 non-null  int64         
 12  supported_languag

## Transformações dos dados

In [22]:
dados_acima_2024.head()

,appid,name,release_date,required_age,price,dlc_count,header_image,windows,mac,linux,...,publishers,positive,negative,average_playtime_forever,median_playtime_forever,discount,peak_ccu,tags,pct_pos_total,num_reviews_total
10,2358720,Black Myth: Wukong,2024-08-19,13,59.99,2,https://shared.akamai.steamstatic.com/store_it...,True,False,False,...,['Game Science'],1098373,37811,0,0,0,35990,"{'Mythology': 9421, 'Action RPG': 7720, 'Actio...",96,825621
15,553850,HELLDIVERS™ 2,2024-02-08,17,39.99,1,https://shared.akamai.steamstatic.com/store_it...,True,False,False,...,['PlayStation Publishing LLC'],743292,234931,0,0,0,48290,"{'Online Co-Op': 939, 'PvE': 827, 'Third-Perso...",76,716489
41,239140,Dying Light,2025-01-27,17,19.99,41,https://shared.akamai.steamstatic.com/store_it...,True,True,True,...,['Techland'],428753,21594,0,0,0,6198,"{'Zombies': 6010, 'Survival Horror': 5732, 'Ho...",95,332069
42,1623730,Palworld,2024-01-18,0,22.49,1,https://shared.akamai.steamstatic.com/store_it...,True,False,False,...,['Pocketpair'],350225,21784,0,0,25,33835,"{'Open World': 1370, 'Survival': 1264, 'Creatu...",94,314689
59,251570,7 Days to Die,2024-07-25,0,29.24,1,https://shared.akamai.steamstatic.com/store_it...,True,True,True,...,['The Fun Pimps Entertainment LLC'],322283,41459,6457,1633,35,25426,"{'Survival': 6898, 'Zombies': 5233, 'Multiplay...",88,249049


In [23]:
list_sep = list()

for tag in dados_acima_2024.tags.values:
    tag_obj = eval(tag)
    if type(tag_obj) == list:
        list_sep.append(eval(tag))
    else:
        list_sep.append(list(eval(tag).keys()))

In [24]:
dados_acima_2024["tags_sep"] = list_sep

In [25]:
dados_acima_2024.drop(["tags"], axis=1, inplace=True)

### Removendo tags vazias

In [26]:
dados_acima_2024[dados_acima_2024["tags_sep"].apply(lambda x: isinstance(x, list) and len(x) == 0)].tags_sep.count()

7741

In [27]:
dados_acima_2024 = dados_acima_2024[~dados_acima_2024["tags_sep"].apply(lambda x: isinstance(x, list) and len(x) == 0)]

In [28]:
dados_acima_2024[dados_acima_2024["tags_sep"].apply(lambda x: isinstance(x, list) and len(x) == 0)].tags_sep.count()

0

### Removendo publishers vazios

In [29]:
dados_acima_2024[dados_acima_2024["publishers"] == '[]'].publishers.count()

16

In [30]:
dados_acima_2024 = dados_acima_2024.replace('[]', np.nan)

In [31]:
dados_acima_2024.dropna(inplace=True)

In [33]:
dados_acima_2024[dados_acima_2024["publishers"] == '[]'].publishers.count()

0

#### Para facilitar nas análises, vai ser considerado apenas o 1° publisher

In [37]:
dados_acima_2024["publishers"].apply(lambda x: eval(x)[0])

10                     Game Science
15       PlayStation Publishing LLC
41                         Techland
73                         Newnight
87          Coffee Stain Publishing
                    ...            
89563                    FalooGames
89566                SerPlanetGames
89582                   30 Parallel
89588                   Clyde Smets
89616                         XuJie
Name: publishers, Length: 6035, dtype: object

In [38]:
dados_acima_2024["publisher"] = dados_acima_2024["publishers"].apply(lambda x: eval(x)[0])

In [39]:
dados_acima_2024["publisher"].head(1)

10    Game Science
Name: publisher, dtype: object

### Removendo dados -1

In [40]:
dados_acima_2024.pct_pos_total.value_counts(ascending=True)

 8         1
 9         1
 13        1
 0         1
 32        1
        ... 
 95      105
 92      112
 90      139
 100     384
-1      3012
Name: pct_pos_total, Length: 85, dtype: int64

In [41]:
dados_acima_2024[dados_acima_2024.pct_pos_total == -1].pct_pos_total.count()

3012

In [42]:
dados_acima_2024[dados_acima_2024.num_reviews_total == -1].num_reviews_total.count()

3012

In [43]:
dados_acima_2024.replace(-1, np.nan, inplace=True)

In [44]:
dados_acima_2024.dropna(inplace=True)

In [45]:
dados_acima_2024[dados_acima_2024.num_reviews_total == -1].num_reviews_total.count()

0

In [46]:
dados_acima_2024.head()

,appid,name,release_date,required_age,price,dlc_count,header_image,windows,mac,linux,...,positive,negative,average_playtime_forever,median_playtime_forever,discount,peak_ccu,pct_pos_total,num_reviews_total,tags_sep,publisher
10,2358720,Black Myth: Wukong,2024-08-19,13.0,59.99,2,https://shared.akamai.steamstatic.com/store_it...,True,False,False,...,1098373,37811,0,0,0,35990,96.0,825621.0,"[Mythology, Action RPG, Action, Souls-like, RP...",Game Science
15,553850,HELLDIVERS™ 2,2024-02-08,17.0,39.99,1,https://shared.akamai.steamstatic.com/store_it...,True,False,False,...,743292,234931,0,0,0,48290,76.0,716489.0,"[Online Co-Op, PvE, Third-Person Shooter, Mult...",PlayStation Publishing LLC
41,239140,Dying Light,2025-01-27,17.0,19.99,41,https://shared.akamai.steamstatic.com/store_it...,True,True,True,...,428753,21594,0,0,0,6198,95.0,332069.0,"[Zombies, Survival Horror, Horror, Online Co-O...",Techland
73,1326470,Sons Of The Forest,2024-02-22,0.0,29.99,0,https://shared.akamai.steamstatic.com/store_it...,True,False,False,...,212182,30041,0,0,0,6723,87.0,215221.0,"[Survival, Open World, Multiplayer, Co-op, Sur...",Newnight
87,526870,Satisfactory,2024-09-10,0.0,39.99,1,https://shared.akamai.steamstatic.com/store_it...,True,False,False,...,217300,6461,4676,1679,0,15097,97.0,189516.0,"[Base-Building, Automation, Open World, Multip...",Coffee Stain Publishing


### Salvando dados tratados

In [47]:
dados_acima_2024.shape

(3022, 25)

In [48]:
dados_acima_2024.to_csv("../data/processed/games_march2025_cleaned.csv", index=False)

In [49]:
dados_acima_2024[:10].to_csv("../data/sample/games_march2025_sample.csv", index=False)